In [1]:
import pandas as pd
import os

In [2]:
# Configure file paths
INPUT_CSV_PATH = '/Users/utkarshumang/Desktop/igleads_no_data.csv'  
OUTPUT_CSV_PATH = 'output.csv'  

In [3]:
# Read input CSV
df = pd.read_csv(INPUT_CSV_PATH)

print(f"Loaded {len(df)} rows")
print(f"\nInput columns: {list(df.columns)}")
df.head()

Loaded 30900 rows

Input columns: ['Business', 'Email', 'Street', 'City', 'State', 'Postcode', 'Country', 'Phone', 'Website', 'Youtube', 'Contact First Name', 'Contact Last Name', 'Contact Role', 'Contact Email', 'Unnamed: 14']


,Business,Email,Street,City,State,Postcode,Country,Phone,Website,Youtube,Contact First Name,Contact Last Name,Contact Role,Contact Email,Unnamed: 14
0,2nd Line Digital Marketing Agency,info@2ndlinemarketing.com,935 Gravier St #1042,New Orleans,LA,70112,United States,(504) 434-0769,https://2ndlinemarketing.com/,NaN,NaN,NaN,NaN,NaN,NaN
1,504 Imaging & Media - Content & Digital Market...,info@504imagingandmedia.com,Poydras St Suit 2900,New Orleans,LA,70163,United States,(504) 475-7511,https://www.504imagingandmedia.com/,https://www.youtube.com/@504ImagingandMedia,NaN,NaN,NaN,NaN,NaN
2,96fx Web Design And Marketing,projects@96fx.com,920 Tchoupitoulas St,New Orleans,LA,70130,United States,(504) 323-5502,http://96fx.com/,NaN,NaN,NaN,NaN,NaN,NaN
3,A La Carte Digital,info@alacarte-digital.com,Tucker Ave,New Orleans,LA,70121,United States,(504) 389-1133,https://alacarte-digital.com/,NaN,NaN,NaN,NaN,NaN,NaN
4,Advidly Marketing & Video Production,info@advidly.com,3014 Dauphine St,New Orleans,LA,70117,United States,(877) 931-6504,http://www.advidly.com/,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
def map_email(row):
    """
    Email mapping logic:
    - If Contact Email and Contact Role both have values -> Contact Email
    - If only Contact Role has value -> Contact Role
    - If both are empty -> Email
    """
    contact_email = row['Contact Email']
    contact_role = row['Contact Role']
    email = row['Email']
    
    # Check if values are not null and not empty strings
    has_contact_email = pd.notna(contact_email) and str(contact_email).strip() != ''
    has_contact_role = pd.notna(contact_role) and str(contact_role).strip() != ''
    
    if has_contact_email and has_contact_role:
        return contact_email
    elif has_contact_role and not has_contact_email:
        return contact_role
    else:
        return email

In [5]:
def map_title(row):
    """
    Title mapping logic:
    - Contact Role if Contact Email has value
    - Otherwise Null
    """
    contact_email = row['Contact Email']
    contact_role = row['Contact Role']
    
    has_contact_email = pd.notna(contact_email) and str(contact_email).strip() != ''
    
    return contact_role if has_contact_email else None

In [6]:
def concatenate_address(row):
    """
    Concatenate address fields with commas for Industry column
    """
    fields = ['Street', 'City', 'State', 'Postcode', 'Country']
    values = [str(row[field]) for field in fields if pd.notna(row[field]) and str(row[field]).strip() != '']
    return ', '.join(values) if values else None

In [7]:
# Create output dataframe
output_df = pd.DataFrame()

# First, map the Email column (we'll use this value for First Name if needed)
output_df['Email'] = df.apply(map_email, axis=1)

# Now map First Name with conditional logic
def map_first_name(row):
    """
    First Name mapping logic:
    - If Contact First Name has value -> Contact First Name
    - If Contact First Name is empty AND (Contact Role OR Contact Email has value) -> use the mapped Email value
    - Otherwise -> Contact First Name (which would be empty)
    """
    contact_first_name = row['Contact First Name']
    contact_email = row['Contact Email']
    contact_role = row['Contact Role']
    
    has_contact_first_name = pd.notna(contact_first_name) and str(contact_first_name).strip() != ''
    has_contact_email = pd.notna(contact_email) and str(contact_email).strip() != ''
    has_contact_role = pd.notna(contact_role) and str(contact_role).strip() != ''
    
    if has_contact_first_name:
        return contact_first_name
    elif (has_contact_role or has_contact_email):
        # Get the already-mapped email value
        return row['_mapped_email']
    else:
        return contact_first_name

# Store mapped email temporarily so we can use it for First Name
df['_mapped_email'] = output_df['Email']

output_df['First Name'] = df.apply(map_first_name, axis=1)
output_df['Last Name'] = df['Contact Last Name']
output_df['Company Name for Emails'] = df['Business']
output_df['Website'] = df['Website']
output_df['Corporate No.'] = df['Phone']
output_df['Company Linkedin Url'] = df['Youtube']

# Conditional mappings
output_df['Title'] = df.apply(map_title, axis=1)
output_df['Industry'] = df.apply(concatenate_address, axis=1)

# No mapping columns (set to None/empty)
no_mapping_columns = [
    'Personal LinkedIn', 'Facebook Url', 'Twitter Url', 'ABOUT_US', 'EBOOK', 
    'COURSES', 'RECENT_BLOG', 'TESTIMONIALS', 'WEBINAR', 'SERVICES', 
    'PODCAST', 'SHOP', 'METADATA', 'Email 1', 'Email 1 Data Point', 
    'Subsequence 1', 'Subsequence 2', 'Subsequence 3', 'Subsequence 2 Data Point',
    'Subsequence 3', 'Subsequence 3 Data Point', 'Subsequence 4', 
    'Subsequence 4 Data Point', 'Email 2', 'Email 3'
]

for col in no_mapping_columns:
    output_df[col] = None

print("Transformation complete!")
print(f"Output shape: {output_df.shape}")

Transformation complete!
Output shape: (30900, 33)


In [8]:
# Display first few rows
output_df.head()

,Email,First Name,Last Name,Company Name for Emails,Website,Corporate No.,Company Linkedin Url,Title,Industry,Personal LinkedIn,...,Email 1 Data Point,Subsequence 1,Subsequence 2,Subsequence 3,Subsequence 2 Data Point,Subsequence 3 Data Point,Subsequence 4,Subsequence 4 Data Point,Email 2,Email 3
0,info@2ndlinemarketing.com,NaN,NaN,2nd Line Digital Marketing Agency,https://2ndlinemarketing.com/,(504) 434-0769,NaN,None,"935 Gravier St #1042, New Orleans, LA, 70112, ...",None,...,None,None,None,None,None,None,None,None,None,None
1,info@504imagingandmedia.com,NaN,NaN,504 Imaging & Media - Content & Digital Market...,https://www.504imagingandmedia.com/,(504) 475-7511,https://www.youtube.com/@504ImagingandMedia,None,"Poydras St Suit 2900, New Orleans, LA, 70163, ...",None,...,None,None,None,None,None,None,None,None,None,None
2,projects@96fx.com,NaN,NaN,96fx Web Design And Marketing,http://96fx.com/,(504) 323-5502,NaN,None,"920 Tchoupitoulas St, New Orleans, LA, 70130, ...",None,...,None,None,None,None,None,None,None,None,None,None
3,info@alacarte-digital.com,NaN,NaN,A La Carte Digital,https://alacarte-digital.com/,(504) 389-1133,NaN,None,"Tucker Ave, New Orleans, LA, 70121, United States",None,...,None,None,None,None,None,None,None,None,None,None
4,info@advidly.com,NaN,NaN,Advidly Marketing & Video Production,http://www.advidly.com/,(877) 931-6504,NaN,None,"3014 Dauphine St, New Orleans, LA, 70117, Unit...",None,...,None,None,None,None,None,None,None,None,None,None


In [9]:
# Check column order
print("Output columns:")
print(list(output_df.columns))

Output columns:
['Email', 'First Name', 'Last Name', 'Company Name for Emails', 'Website', 'Corporate No.', 'Company Linkedin Url', 'Title', 'Industry', 'Personal LinkedIn', 'Facebook Url', 'Twitter Url', 'ABOUT_US', 'EBOOK', 'COURSES', 'RECENT_BLOG', 'TESTIMONIALS', 'WEBINAR', 'SERVICES', 'PODCAST', 'SHOP', 'METADATA', 'Email 1', 'Email 1 Data Point', 'Subsequence 1', 'Subsequence 2', 'Subsequence 3', 'Subsequence 2 Data Point', 'Subsequence 3 Data Point', 'Subsequence 4', 'Subsequence 4 Data Point', 'Email 2', 'Email 3']


In [10]:
# Define desired column order
desired_order = [
    'First Name', 'Last Name', 'Title', 'Company Name for Emails', 'Email',
    'Personal LinkedIn', 'Website', 'Corporate No.', 'Industry', 
    'Company Linkedin Url', 'Facebook Url', 'Twitter Url', 'ABOUT_US', 
    'EBOOK', 'COURSES', 'RECENT_BLOG', 'TESTIMONIALS', 'WEBINAR', 
    'SERVICES', 'PODCAST', 'SHOP', 'METADATA', 'Email 1', 
    'Email 1 Data Point', 'Subsequence 1', 'Subsequence 2', 'Subsequence 3',
    'Subsequence 2 Data Point', 'Subsequence 3', 'Subsequence 3 Data Point',
    'Subsequence 4', 'Subsequence 4 Data Point', 'Email 2', 'Email 3'
]

output_df = output_df[desired_order]
print("Columns reordered successfully!")

Columns reordered successfully!


In [11]:
# Save to CSV
output_df.to_csv(OUTPUT_CSV_PATH, index=False)
print(f"✓ Output saved to: {OUTPUT_CSV_PATH}")
print(f"✓ Total rows: {len(output_df)}")
print(f"✓ Total columns: {len(output_df.columns)}")

✓ Output saved to: output.csv
✓ Total rows: 30900
✓ Total columns: 34


In [12]:
# Check for any null values in key columns
key_columns = ['First Name', 'Last Name', 'Email', 'Company Name for Emails']
print("Null value counts in key columns:")
print(output_df[key_columns].isnull().sum())

Null value counts in key columns:
First Name                 29987
Last Name                  30045
Email                          0
Company Name for Emails        0
dtype: int64


In [13]:
# Sample some rows to verify transformations
print("Sample rows:")
output_df[['First Name', 'Last Name', 'Title', 'Email', 'Industry']].sample(min(5, len(output_df)))

Sample rows:


,First Name,Last Name,Title,Email,Industry
4988,NaN,NaN,None,info@excelrainman.com,"1407 W Grand Ave, Chicago, Il, 60642, United S..."
13789,NaN,NaN,None,email@bulkmatic.com,"7029 9 Commonwealth Ave, Jacksonville, FL, 322..."
18802,NaN,NaN,None,contact@altitude.com,"150 Se 25th Rd, Miami, FL, 33129, United States"
23482,NaN,NaN,None,info@thebrooklynmonarch.com,"3205 Orleans Ave, New Orleans, LA, 70119, Unit..."
29782,NaN,NaN,None,info@americanhealthimaging.com,"7972 N Oracle Rd, Oro Valley, AZ, 85704, Unite..."
